<a href="https://colab.research.google.com/github/sushantkai/AI-BASED-RESUME-EVALUATION-TOOL/blob/main/Assessment_1_English%E2%80%93Hindi_Dataset_Processing_and_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assessment 1: English–Hindi Dataset Processing and Analysis


In [1]:
# Cell 1: Install dependencies
!pip install openpyxl -q

# Upload your zip file
from google.colab import files
uploaded = files.upload()  # Upload: Dataset_English_Hindi_csv.zip

Saving Dataset_English_Hindi.csv.zip to Dataset_English_Hindi.csv.zip


In [3]:
# Cell 2: Extract and load dataset
import zipfile, pandas as pd, os

# Extract zip
with zipfile.ZipFile('/content/Dataset_English_Hindi.csv.zip', 'r') as z:
    z.extractall('dataset/')

csv_file = [f for f in os.listdir('dataset/') if f.endswith('.csv')][0]
df = pd.read_csv(f'dataset/{csv_file}')
df.columns = ['English', 'Hindi']
df = df.dropna()
print(f"Total rows loaded: {len(df):,}")
df.head()

Total rows loaded: 130,162


,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [4]:
# Cell 3: Word count analysis and filtering
import re

def clean_text(text):
    if not isinstance(text, str):
        return str(text)
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

df['English'] = df['English'].apply(clean_text)
df['Hindi']   = df['Hindi'].apply(clean_text)

# Compute word counts
df['WC_English'] = df['English'].str.split().str.len()
df['WC_Hindi']   = df['Hindi'].str.split().str.len()

# Filter: 5–50 words in both languages
df_filtered = df[
    (df['WC_English'] >= 5) & (df['WC_English'] <= 50) &
    (df['WC_Hindi']   >= 5) & (df['WC_Hindi']   <= 50)
].copy()

# Word count difference and filter: -10 to +10
df_filtered['WC_Difference'] = df_filtered['WC_English'] - df_filtered['WC_Hindi']
df_final = df_filtered[
    (df_filtered['WC_Difference'] >= -10) &
    (df_filtered['WC_Difference'] <= 10)
].reset_index(drop=True)

print(f"Rows after filtering: {len(df_final):,}")
df_final.head()

Rows after filtering: 102,970


,English,Hindi,WC_English,WC_Hindi,WC_Difference
0,He has a hat on.,उसने टोपी पहनी हुई है।,5,5,0
1,He came to see me.,वह मुझसे मिलने आया था।,5,5,0
2,He is after a job.,वह किसी नौकरी के पीछे पड़ा है।,5,7,-2
3,I can drive a car.,मैं गाड़ी चला सकता हूँ।,5,5,0
4,I work for a bank.,मैं बैंक में काम करता हूँ।,5,6,-1


In [5]:
# Cell 4: Create and download Excel file
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Cleaned Dataset"

# Header styling
header_font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
header_fill = PatternFill('solid', start_color='1F4E79')
center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
left_align   = Alignment(horizontal='left',   vertical='center', wrap_text=True)

headers    = ['English Sentences', 'Hindi Sentences', 'Word Count (English)', 'Word Count (Hindi)', 'Word Count Difference (Eng - Hin)']
col_widths = [60, 60, 22, 20, 35]

for i, (h, w) in enumerate(zip(headers, col_widths), 1):
    cell = ws.cell(row=1, column=i, value=h)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = center_align
    ws.column_dimensions[get_column_letter(i)].width = w

ws.row_dimensions[1].height = 35

light_blue = PatternFill('solid', start_color='DCE6F1')
white_fill = PatternFill('solid', start_color='FFFFFF')

print("Writing rows... (may take a moment)")
for idx, row in df_final.iterrows():
    r    = idx + 2
    fill = light_blue if idx % 2 == 0 else white_fill

    ws.cell(row=r, column=1, value=row['English']).alignment      = left_align
    ws.cell(row=r, column=2, value=row['Hindi']).alignment        = left_align
    ws.cell(row=r, column=3, value=int(row['WC_English'])).alignment = center_align
    ws.cell(row=r, column=4, value=int(row['WC_Hindi'])).alignment   = center_align
    ws.cell(row=r, column=5, value=f'=C{r}-D{r}').alignment         = center_align

    for c in range(1, 6):
        ws.cell(row=r, column=c).fill = fill

ws.freeze_panes = 'A2'

# Summary sheet
ws2 = wb.create_sheet("Summary")
ws2['A1'] = 'Assessment 1 – Summary Statistics'
ws2['A1'].font = Font(name='Arial', bold=True, size=14, color='1F4E79')

summary = [
    ('Total rows (original dataset)',            len(df)),
    ('Rows after all filters applied',           len(df_final)),
    ('Avg Word Count (English)',                 round(df_final['WC_English'].mean(), 2)),
    ('Avg Word Count (Hindi)',                   round(df_final['WC_Hindi'].mean(), 2)),
    ('Min / Max WC Difference',                  f"{int(df_final['WC_Difference'].min())} / {int(df_final['WC_Difference'].max())}"),
]
for i, (label, value) in enumerate(summary, 3):
    ws2.cell(row=i, column=1, value=label).font = Font(name='Arial', bold=True)
    ws2.cell(row=i, column=2, value=value)

ws2.column_dimensions['A'].width = 45
ws2.column_dimensions['B'].width = 25

output_path = 'Assessment1_English_Hindi_Dataset.xlsx'
wb.save(output_path)
print(f"✅ Saved: {output_path}")

# Auto-download
from google.colab import files
files.download(output_path)

Writing rows... (may take a moment)
✅ Saved: Assessment1_English_Hindi_Dataset.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Assessment 2: Translation with LLM

In [12]:
# Cell 1: Install Google Gemini SDK
!pip install google-generativeai -q

In [19]:
# Cell 2: Set Gemini API key
import google.generativeai as genai
from google.colab import userdata

# Option A: Colab Secrets (recommended)
# Click 🔑 in left panel → Add secret → Name: GEMINI_API_KEY → paste key
try:
    api_key = userdata.get('Gem')
except:
    api_key = "AIza..."   # ← paste your key directly here if Secrets not set up

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-1.5-flash")  # free tier model
print("✅ Gemini client ready")

✅ Gemini client ready


In [20]:
# Cell 3: Pick 100 sentences from Assessment 1 filtered data
# (Run Assessment 1 cells first, or load your saved CSV)

sample_100 = df_final.sample(100, random_state=42)[['English', 'Hindi']].reset_index(drop=True)
print(f"Selected {len(sample_100)} sentences")
sample_100.head(3)

Selected 100 sentences


,English,Hindi
0,"because it's vital, not just to ourselves,","क्योंकि यह महत्वपूर्ण है न केवल खुद को,"
1,The untouchables were expected to announce the...,अछूतों से अपेक्षा रहती थी कि नगर में प्रवेश कर...
2,With some of our earliest specimens dating fro...,हमारे कुछ पूर्ववर्ती नमूने 1400 के उत्तरार्द्ध...


In [21]:
# Cell 4: Translate English → Hindi using Gemini (batches of 10)
import time, json, re

BATCH_SIZE = 10
llm_translations = []
sentences = sample_100['English'].tolist()

def translate_batch(batch):
    prompt = (
        "Translate the following English sentences to Hindi. "
        "Return ONLY a JSON array of strings with the Hindi translations in the same order. "
        "No explanation, no markdown, just the raw JSON array.\n\nSentences:\n"
        + "\n".join(f"{i+1}. {s}" for i, s in enumerate(batch))
    )
    response = model.generate_content(prompt)
    text  = response.text
    clean = re.sub(r'```json|```', '', text).strip()
    return json.loads(clean)

print(f"Translating {len(sentences)} sentences in batches of {BATCH_SIZE}...\n")

for i in range(0, len(sentences), BATCH_SIZE):
    batch = sentences[i:i + BATCH_SIZE]
    try:
        translated = translate_batch(batch)
        for j, t in enumerate(translated):
            llm_translations.append({
                'No.'                   : i + j + 1,
                'English Sentence'      : batch[j],
                'Reference Hindi'       : sample_100.loc[i + j, 'Hindi'],
                'LLM Hindi Translation' : t
            })
        print(f"  ✅ Batch {i//BATCH_SIZE + 1}/{(len(sentences)-1)//BATCH_SIZE + 1} done  ({i + len(batch)}/{len(sentences)} sentences)")
    except Exception as e:
        print(f"  ❌ Batch {i//BATCH_SIZE + 1} failed: {e}")
        for j, s in enumerate(batch):
            llm_translations.append({
                'No.'                   : i + j + 1,
                'English Sentence'      : s,
                'Reference Hindi'       : sample_100.loc[i + j, 'Hindi'],
                'LLM Hindi Translation' : 'ERROR'
            })
    time.sleep(1)  # avoid rate limit

df_translated = pd.DataFrame(llm_translations)
print(f"\n✅ Translation complete! {len(df_translated)} rows")
df_translated.head(5)

Translating 100 sentences in batches of 10...



  ❌ Batch 1 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
  ❌ Batch 2 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  ❌ Batch 3 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
  ❌ Batch 4 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


  ❌ Batch 5 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
  ❌ Batch 6 failed: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


  ❌ Batch 7 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


  ❌ Batch 8 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
  ❌ Batch 9 failed: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


  ❌ Batch 10 failed: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

✅ Translation complete! 100 rows


,No.,English Sentence,Reference Hindi,LLM Hindi Translation
0,1,"because it's vital, not just to ourselves,","क्योंकि यह महत्वपूर्ण है न केवल खुद को,",ERROR
1,2,The untouchables were expected to announce the...,अछूतों से अपेक्षा रहती थी कि नगर में प्रवेश कर...,ERROR
2,3,With some of our earliest specimens dating fro...,हमारे कुछ पूर्ववर्ती नमूने 1400 के उत्तरार्द्ध...,ERROR
3,4,Our common mole cricket Glryllotalpa prepares ...,सामान्य छछुंद झींगुर ग्राइलोटैल्पा एक भूमिगत श...,ERROR
4,5,I ought to explain that I have no idea what wa...,मुझे नहीं पता कि उस समय पर क्या हो रहा था,ERROR
